[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/milioe/casos-ia-ibero-diplomado/blob/main/modulo_4/11_RAG_fundamentos.ipynb)


# RAG — fundamentos con un PDF real

En el **09** buscamos en una lista con embeddings. Aquí hacemos **RAG** (*Retrieval-Augmented Generation*) en la parte de **recuperación**:

1. Leer un PDF desde una URL con **`pypdf`** (en memoria, sin guardar archivo).
2. Un **chunk = una página**.
3. Embeddings con Gemini (igual que el 09).
4. Índice **FAISS** en memoria y consultas con preguntas.

Documento: [The State of Enterprise AI — 2025 Report (OpenAI)](https://cdn.openai.com/pdf/7ef17d82-96bf-4dd1-9df2-228f7f377a29/the-state-of-enterprise-ai_2025-report.pdf)

**Este notebook está pensado para correr en Google Colab** (usa el panel de *Secrets* para la llave), no en local.

## Setup — instalar librerías

- `google-genai`, `numpy`, `pypdf`, `faiss-cpu`, `requests`

In [11]:
%pip install -q google-genai numpy pypdf faiss-cpu requests

Note: you may need to restart the kernel to use updated packages.


In [12]:
import io
import os
import textwrap

import faiss
import numpy as np
import requests
from google import genai
from pypdf import PdfReader

## 1 — ¿Qué es RAG?

**RAG** = recuperar trozos del documento parecidos a la pregunta y (en un sistema completo) pasárselos a un LLM para responder con evidencia.

Hoy solo **encontramos la página** correcta. Los datos viven en variables de Python: `paginas`, `embeddings`, `indice`.


## 2 — Clave API (Google AI Studio, desde Colab)

1. Entra a [Google AI Studio → API keys](https://aistudio.google.com/apikey) y crea una clave.
2. En Colab, abre el panel de **Secrets** (ícono de llave 🔑 en la barra lateral izquierda) y guárdala con el nombre `GOOGLE_API_KEY`. Actívala para este notebook.

In [13]:
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
API_KEY = os.environ["GOOGLE_API_KEY"].strip()

cliente = genai.Client(api_key=API_KEY)
MODELO_EMBEDDING = "gemini-embedding-001"

print(f"Cliente listo. Modelo: {MODELO_EMBEDDING}")

Cliente listo. Modelo: gemini-embedding-001


## 3 — Leer el PDF desde la URL

Descargamos los bytes en memoria y `pypdf` los lee directo (no hace falta guardar el PDF en disco). **Un chunk = una página.**


In [14]:
URL_PDF = (
    "https://cdn.openai.com/pdf/7ef17d82-96bf-4dd1-9df2-228f7f377a29/"
    "the-state-of-enterprise-ai_2025-report.pdf"
)

resp = requests.get(URL_PDF, timeout=120)
resp.raise_for_status()
reader = PdfReader(io.BytesIO(resp.content))

paginas = []
for i, page in enumerate(reader.pages):
    paginas.append({
        "pagina": i + 1,
        "texto": (page.extract_text() or "").strip(),
    })

print(f"Páginas leídas: {len(paginas)}")
print(paginas[0]["texto"][:250], "...")

Páginas leídas: 24
2025 REPORT
The state of 
enterprise AI ...


## 4 — Ver una página por número

Cambia `NUM_PAGINA` y ejecuta.


In [15]:
NUM_PAGINA = 2


def mostrar_pagina(n):
    p = paginas[n - 1]
    print(f"=== Página {p['pagina']} / {len(paginas)} ===\n")
    print(textwrap.fill(p["texto"] or "(sin texto)", width=88))


mostrar_pagina(NUM_PAGINA)

=== Página 2 / 24 ===

The state of enterprise AI  |  2025 Report2 Foreword At OpenAI, our mission is to ensure
that  artificial intelligence benefits all of humanity,   and helping enterprises solve
problems is   central to this mission.  The majority of economically valuable activity
takes place inside  organizations, where innovation translates directly into improved
outcomes  for workers, customers, and other stakeholders. Enterprise problems also
present the hardest technical challenges for frontier intelligence, requiring
reliability, safety, and security at scale. The revenue generated from solving  these
problems can help fund broad, free access to powerful AI for hundreds  of millions of
people worldwide.  For much of the past three years, the visible impact of AI has been
most  apparent among consumers. However, the history of general purpose
technologies—from steam engines to semiconductors—shows that   significant economic
value is created after firms translate underlying  

## 5 — Embeddings + índice FAISS (todo en memoria)

- `embeddings` — matriz NumPy (una fila por página).
- `indice` — objeto FAISS para buscar vecinos cercanos.
- `paginas_indexadas` — misma lista de páginas, en el mismo orden que las filas del índice.

Usamos `IndexFlatIP`: compara todas las páginas (vale bien con ~24 vectores). Con vectores normalizados, el producto interno ≈ similitud coseno.


In [16]:
def embed_textos(textos):
    if isinstance(textos, str):
        textos = [textos]
    resp = cliente.models.embed_content(model=MODELO_EMBEDDING, contents=textos)
    return np.array([e.values for e in resp.embeddings], dtype=np.float32)


paginas_indexadas = [p for p in paginas if len(p["texto"]) > 50]
textos = [p["texto"] for p in paginas_indexadas]

print("Calculando embeddings...")
embeddings = embed_textos(textos)
faiss.normalize_L2(embeddings)

indice = faiss.IndexFlatIP(embeddings.shape[1])
indice.add(embeddings)

print(f"embeddings.shape = {embeddings.shape}")
print(f"vectores en índice = {indice.ntotal}")

Calculando embeddings...
embeddings.shape = (23, 3072)
vectores en índice = 23


## 6 — Consultar

Pregunta → embedding → FAISS devuelve las páginas más parecidas. Puedes preguntar en **español** sobre un PDF en **inglés**.


In [17]:
pregunta = "¿Cuál es la manera más usada para construir aplicaciones de IA?"
# pregunta = "¿En qué usan más las empresas la API de OpenAI?"
# pregunta = "¿Qué sectores están adoptando más rápido la inteligencia artificial?"
# pregunta = "¿Cuánto tiempo dicen ahorrar los trabajadores cuando usan ChatGPT en el trabajo?"
# pregunta = "¿Para qué están usando las empresas los GPTs personalizados y los Projects?"
# pregunta = "¿Qué países están creciendo más rápido en adopción de IA empresarial?"
k = 3

vec_pregunta = embed_textos(pregunta)
faiss.normalize_L2(vec_pregunta)
scores, idxs = indice.search(vec_pregunta, k)

print(f"Pregunta: {pregunta}\n")
for rank, (idx, score) in enumerate(zip(idxs[0], scores[0]), start=1):
    p = paginas_indexadas[idx]
    print("=" * 88)
    print(f"#{rank}  Página {p['pagina']}  —  score = {score:.4f}")
    print(textwrap.fill(p["texto"][:1000], width=88))
    print()

Pregunta: ¿Cuál es la manera más usada para construir aplicaciones de IA?

#1  Página 11  —  score = 0.6544
The state of enterprise AI  |  2025 Report11 Pace of acceleration varies based on
industry and geography The API is most commonly used to build and scale customer-facing
applications (e.g., in-product assistants, search, and automation), particularly  by
technology companies. But usage is diversifying: customer service and  content
generation now represent approximately 20% of API activity, and  non-technology firm API
use has grown 5x year-over-year. T aken together,   this pattern suggests adoption is
expanding beyond technology-led product  embedding toward a broader set of operational
and workflow deployments  across industries. T echnology companies are  using the API at
a rate 5x  higher year-over-year as they  scale external, customer- facing applications.
They also  lead in coding workflows,  where frontier models such as  Codex are
accelerating  software development. T o

## 7 — Tipos de índices (FAISS y bases vectoriales)

En este cuaderno usamos **`IndexFlatIP`** + vectores **normalizados** (`faiss.normalize_L2`):

- **Flat** = compara la pregunta contra **todos** los vectores (búsqueda exacta, sin atajos).
- **IP** (*inner product*, producto interno) = con vectores unitarios equivale a **similitud coseno**.

Con ~24 páginas es instantáneo. Con **millones** de vectores hace falta otro tipo de índice.

### Familias principales en FAISS

| Índice | Idea | Ventajas | Desventajas | Cuándo usarlo |
|--------|------|----------|-------------|----------------|
| **Flat** (`IndexFlatL2`, `IndexFlatIP`) | Fuerza bruta: revisa todo | Resultado **exacto**; fácil de entender | Lento y pesado si hay millones de vectores | Miles o menos de vectores (como este PDF) |
| **IVF** (`IndexIVFFlat`, …) | Parte el espacio en **clústeres**; solo busca en los más prometedores | Mucho más rápido a gran escala | Aproximado: puede **perder** el vecino correcto; hay que **entrenar** el índice | Cientos de miles / millones de documentos |
| **HNSW** (`IndexHNSWFlat`, …) | Grafo de vecinos: salta de nodo en nodo | Muy rápido en consulta; buen equilibrio calidad/velocidad | Aproximado; más RAM; parámetros que afinar | Producción con muchas consultas por segundo |
| **PQ** (*Product Quantization*) | Comprime cada vector en códigos cortos | Poca memoria en disco/RAM | Pierde precisión; peor ranking fino | Cuando no caben todos los embeddings en RAM |

A menudo se **combinan** (ej. `IVF + PQ`: rápido y compacto).

### L2 vs IP (métrica)

| Métrica | Qué mide | En FAISS |
|---------|----------|----------|
| **L2** | Distancia euclidiana (más lejos = menos parecido) | `IndexFlatL2` |
| **IP / coseno** | Ángulo entre vectores (con normalización ≈ coseno) | `IndexFlatIP` + `normalize_L2` |

Para embeddings de texto suele ir mejor **coseno** (IP + normalizar), que es lo que hicimos.

### ¿Y Pinecone, Chroma, pgvector…?

Son **bases o servicios** que por dentro también usan índices (a veces HNSW u otros). La lógica es la misma:

1. Guardas vectores + metadatos (texto, número de página).
2. Consultas con el embedding de la pregunta.
3. Recuperas los **k** más cercanos.

FAISS es la **librería** (tú montas todo). Chroma/Pinecone te dan API y persistencia listas.

### Regla práctica para elegir

- **Menos de ~10 000 vectores** → `IndexFlatIP` (este notebook).
- **10⁵ – 10⁷** → IVF o HNSW.
- **Poca RAM** → PQ o servicio gestionado.
- **No puedes perder el mejor match** → Flat o HNSW conservador; evita IVF muy agresivo.

**Cierre.** Variables del pipeline: `paginas_indexadas`, `embeddings`, `indice`. Siguiente paso: pasar las páginas recuperadas a un LLM para la respuesta final. Prueba otras preguntas cambiando `pregunta` en la sección 6.


## 8 — Subir los vectores a Chroma Cloud

FAISS vive en la memoria del cuaderno: si cierras Colab, se pierde. **Chroma Cloud** guarda los mismos vectores de Gemini en la nube y además permite una búsqueda **híbrida**: vectores densos (significado) + dispersos Splade (palabras clave), combinados con **RRF** (*Reciprocal Rank Fusion*).

Necesitas en los *Secrets* de Colab: `CHROMA_API_KEY`, `CHROMA_TENANT` y `CHROMA_DATABASE`.

In [18]:
%pip install -q chromadb

Note: you may need to restart the kernel to use updated packages.


In [19]:
import chromadb
from chromadb import K, Knn, Rrf, Schema, Search, SparseVectorIndexConfig, VectorIndexConfig
from chromadb.execution.expression import GroupBy, MinK
from chromadb.utils.embedding_functions import ChromaCloudSpladeEmbeddingFunction

os.environ["CHROMA_API_KEY"] = userdata.get("CHROMA_API_KEY")  # lo usa también Splade

cliente_chroma = chromadb.CloudClient(
    tenant=userdata.get("CHROMA_TENANT"),
    database=userdata.get("CHROMA_DATABASE"),
    api_key=os.environ["CHROMA_API_KEY"],
)

# Denso: nuestros vectores de Gemini (los subimos ya calculados, coseno).
# Disperso: Splade, lo calcula Chroma en el servidor a partir del texto.
schema = Schema()
schema.create_index(config=VectorIndexConfig(space="cosine"))
schema.create_index(
    config=SparseVectorIndexConfig(
        source_key=K.DOCUMENT,
        embedding_function=ChromaCloudSpladeEmbeddingFunction(),
    ),
    key="sparse_embedding",
)

coleccion = cliente_chroma.get_or_create_collection(
    name="rag_state_of_enterprise_ai_2025", schema=schema
)
print(f"Colección lista: {coleccion.name}")

NameError: name 'userdata' is not defined

In [ ]:
MAX_BYTES = 12_000  # Chroma acepta máx. 16 KiB por documento


def chunks_por_lineas(texto, max_bytes=MAX_BYTES):
    actual, tam, salida = [], 0, []
    for linea in texto.splitlines():
        n = len(linea.encode()) + 1
        if actual and tam + n > max_bytes:
            salida.append("\n".join(actual))
            actual, tam = [], 0
        actual.append(linea)
        tam += n
    if actual:
        salida.append("\n".join(actual))
    return salida


ids, docs, metas, vecs = [], [], [], []
for p, vec in zip(paginas_indexadas, embeddings):
    chunks = chunks_por_lineas(p["texto"])
    # una página normal cabe en un chunk: reusamos su vector; si no, re-embebemos los chunks
    vectores = [vec] if len(chunks) == 1 else embed_textos(chunks)
    for i, (chunk, v) in enumerate(zip(chunks, vectores)):
        ids.append(f"pagina-{p['pagina']}-chunk-{i}")
        docs.append(chunk)
        metas.append({"doc_id": f"pagina-{p['pagina']}", "pagina": p["pagina"], "chunk_index": i})
        vecs.append(v.tolist())

for j in range(0, len(ids), 100):
    s = slice(j, j + 100)
    coleccion.upsert(ids=ids[s], documents=docs[s], metadatas=metas[s], embeddings=vecs[s])

print(f"Subidos {len(ids)} chunks. Total en Chroma: {coleccion.count()}")

### Consultar Chroma (búsqueda híbrida)

`Knn` denso usa el vector de la pregunta (Gemini); `Knn` disperso usa el texto. `Rrf` los fusiona (70 % significado, 30 % palabras clave) y `GroupBy` deja solo el mejor chunk por página.

In [ ]:
pregunta = "¿Cuál es la manera más usada para construir aplicaciones de IA?"
k = 3

vec_pregunta = embed_textos(pregunta)
faiss.normalize_L2(vec_pregunta)

denso = Knn(query=vec_pregunta[0].tolist(), key="#embedding", return_rank=True, limit=50)
disperso = Knn(query=pregunta, key="sparse_embedding", return_rank=True, limit=50)

busqueda = (
    Search()
    .rank(Rrf(ranks=[denso, disperso], weights=[0.7, 0.3], k=60))
    .group_by(GroupBy(keys=K("doc_id"), aggregate=MinK(keys=K.SCORE, k=1)))
    .limit(k)
    .select(K.DOCUMENT, K.SCORE, "pagina")
)

print(f"Pregunta: {pregunta}\n")
for rank, fila in enumerate(coleccion.search(busqueda).rows()[0], start=1):
    print("=" * 88)
    print(f"#{rank}  Página {fila['metadata']['pagina']}  —  score RRF = {fila['score']:.4f}")
    print(textwrap.fill(fila["document"][:1000], width=88))
    print()